# 06 - MySQL

Read this notebook and paste each Bash block into an Ubuntu terminal, in order. Keep the same terminal for the commands in this notebook. These are shell instructions; do not run them in a Python kernel.

## Open the project

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



Service: `mysql:3306`. StatefulSet/pod: `mysql` / `mysql-0`. PVC: `mysql-data`, 5 GiB, container volume `data`, mounted at `/var/lib/mysql`. Secret: `mysql-credentials`. Initial database: `training`; example database/table: `test_database.test_table`. The example uses root only to create the database and grant access; table operations use `team`.

## Apply MySQL

```bash
kubectl apply -f mysql/mysql.yaml
```



## Wait for MySQL

```bash
kubectl rollout status statefulset/mysql --timeout=600s
```



## Inspect MySQL storage

```bash
kubectl get pod mysql-0
kubectl get service mysql
kubectl get pvc mysql-data
```



## Create test_database and grant access

```bash
kubectl exec -i mysql-0 -- sh -ec 'MYSQL_PWD="$MYSQL_ROOT_PASSWORD" exec mysql --protocol=TCP -h127.0.0.1 -uroot' <<'SQL'
CREATE DATABASE IF NOT EXISTS test_database;
GRANT ALL ON test_database.* TO 'team'@'%';
SQL
```



## Create and write test_table

```bash
kubectl exec -i mysql-0 -- sh -ec 'MYSQL_PWD="$MYSQL_PASSWORD" exec mysql --protocol=TCP -h mysql -u "$MYSQL_USER" test_database' <<'SQL'
CREATE TABLE IF NOT EXISTS test_table (id INT PRIMARY KEY, message VARCHAR(100) NOT NULL);
INSERT INTO test_table VALUES (1, 'workspace database example') ON DUPLICATE KEY UPDATE message = VALUES(message);
SELECT CURRENT_USER(), DATABASE();
SELECT * FROM test_table;
SQL
```

Expect `team`, `test_database`, and row 1 with `workspace database example`.

## Check first-start logs and metrics

```bash
python3 validations/check-observability.py --components mysql
```

Monitoring was started in notebook 02. This checks the actual first-start daemon logs and CPU/memory samples without restarting the service to generate logs.


## Stop this service

```bash
kubectl scale statefulset/mysql --replicas=0
kubectl wait --for=delete pod -l app=mysql --timeout=180s
```

This retains its Service and storage. Run after finishing the examples.

## Resume this service

```bash
kubectl scale statefulset/mysql --replicas=1
kubectl rollout status statefulset/mysql --timeout=600s
```

